## **Text recognition of a baseline**
We use a base [model](https://github.com/Jean-Baptiste-Camps/FROC-MSS/tree/master) trained in Old French and Old Occitan manuscripts besides kraken to have the first approximation to the verbatim transcription

In [ ]:
import os
import subprocess
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
PROJECT_ROOT = os.environ["PROJECT_ROOT"]

In [38]:
#input_path = os.path.join(PROJECT_ROOT, "data/processed/filtered_images/20260510_160650/original/kept/05_garde_001/05_garde_001_line_0.png")
input_path = os.path.join(PROJECT_ROOT, "data/raw/original_manuscript/reproduction14453_100/15 - f. 010v - 011.jpg")
txt_name = os.path.splitext(os.path.basename(input_path))[0] + ".txt"
txt_path = os.path.join(PROJECT_ROOT, "tests/ocr", txt_name)
os.makedirs(os.path.dirname(txt_path), exist_ok=True)
ocr_model = os.path.join(PROJECT_ROOT, "models/ocr/catmus-medieval.mlmodel")

## **1. Transcription for a complete image**
It performs again the segmentation an on top of that it runs the OCR model

In [73]:
import os
import warnings
import inspect
from dotenv import load_dotenv
from PIL import Image
from kraken import blla, rpred

load_dotenv()
PROJECT_ROOT = os.environ["PROJECT_ROOT"]

input_path = os.path.join(PROJECT_ROOT, "data/raw/original_manuscript/reproduction14453_100/15 - f. 010v - 011.jpg")
output_dir = os.path.join(PROJECT_ROOT, "tests/ocr")
os.makedirs(output_dir, exist_ok=True)
ocr_model = os.path.join(PROJECT_ROOT, "models/ocr/catmus-medieval.mlmodel")

# Find loader
load_model_fn = None
for mod_name in ["kraken.lib.ml", "kraken.lib.models"]:
    try:
        mod = __import__(mod_name, fromlist=["*"])
        candidates = [name for name, obj in inspect.getmembers(mod) if callable(obj) and "load" in name.lower()]
        if candidates:
            load_model_fn = getattr(mod, candidates[0])
            break
    except ImportError: continue

if not load_model_fn: raise RuntimeError("Loader not found")

warnings.filterwarnings("ignore", category=RuntimeWarning)

print("📖 Loading image...")
im = Image.open(input_path).convert("RGB")

print("🔪 Running segmentation...")
seg = blla.segment(im)
print(f"   Found {len(seg.lines)} lines.")

print("🤖 Running OCR...")
try:
    model = load_model_fn(ocr_model, device="cpu")
except TypeError:
    model = load_model_fn(ocr_model)

# Get predictions
preds = list(rpred.rpred(model, im, seg))
print(f"   Got {len(preds)} predictions.")

# 🔍 DEBUG: Inspect the first prediction
if preds:
    p = preds[0]
    print(f"   🔍 Type of prediction: {type(p)}")
    print(f"   🔍 Dir of prediction: {[x for x in dir(p) if not x.startswith('_')]}")
    # Try to access text safely
    if hasattr(p, 'text'): print(f"   🔍 p.text = '{p.text}'")
    if hasattr(p, 'prediction'): print(f"   🔍 p.prediction = '{p.prediction}'")

# Attach text
for i, (line, pred) in enumerate(zip(seg.lines, preds)):
    # Try multiple attribute names common in Kraken versions
    text_val = getattr(pred, 'text', None)
    if text_val is None:
        text_val = getattr(pred, 'prediction', None)
    if text_val is None and isinstance(pred, dict):
        text_val = pred.get('text')
    
    # Fallback to string representation if all else fails
    if text_val is None:
        text_val = str(pred)
        
    line.text = text_val

# Save
base_name = os.path.splitext(os.path.basename(input_path))[0]
txt_path = os.path.join(output_dir, f"{base_name}.txt")

with open(txt_path, "w", encoding="utf-8") as f:
    for line in seg.lines:
        # Write even if empty to debug, or filter if you prefer
        f.write(f"{line.text}\n") 

print(f"✅ Done! Check: {txt_path}")

📖 Loading image...
🔪 Running segmentation...
   Found 202 lines.
🤖 Running OCR...
   Got 202 predictions.
   🔍 Type of prediction: <class 'kraken.containers.BaselineOCRRecord'>
   🔍 Dir of prediction: ['base_dir', 'baseline', 'boundary', 'confidences', 'cuts', 'display_order', 'id', 'imagename', 'logical_order', 'prediction', 'regions', 'split', 'tags', 'text', 'type']
   🔍 p.text = 'None'
   🔍 p.prediction = 'que tengua le sieu ale entro que sia fo'
✅ Done! Check: /Users/camilabermudezvalderrama/Documents/LMU-STATISTICS & DATA SCIENCE MASTER/SS2026/Thesis/OCC_HTR/tests/ocr/15 - f. 010v - 011.txt


## **2. Transcribe just 1 text line image**
In this case it creates a fake base line for the OCR model, in the implemented function I used the scaled baseline we had during the segmentation

In [103]:
import os
from dotenv import load_dotenv
from PIL import Image
from kraken import rpred
from kraken.containers import BaselineLine, Segmentation
from kraken.lib import models

load_dotenv()
PROJECT_ROOT = os.environ["PROJECT_ROOT"]

input_path = os.path.join(PROJECT_ROOT, "data/processed/filtered_images/20260510_160650/original/kept/05_garde_001/05_garde_001_line_15.png")
model_path = os.path.join(PROJECT_ROOT, "models/ocr/catmus-medieval.mlmodel")

# ── Load image as RGB (model expects RGB, not grayscale) ──
im = Image.open(input_path).convert("RGB")
w, h = im.size
print("Image size:", (w, h))

# ── Load model ──
model = models.load_any(model_path, device="cpu")

# ── Boundary: stay strictly inside bounds (w-1, h-1) ──
baseline_y = max(1, min(int(h * 0.75), h - 2))

line = BaselineLine(
    id="0",
    baseline=[(0, baseline_y), (w - 1, baseline_y)],
    boundary=[
        (0,     0    ),
        (w - 1, 0    ),
        (w - 1, h - 1),
        (0,     h - 1),
    ]
)

seg = Segmentation(
    type="baselines",
    imagename=input_path,
    text_direction="horizontal-lr",
    script_detection=False,
    lines=[line]
)

# ── Run OCR ──
preds = list(rpred.rpred(model, im, seg))

print("\n==================== RESULT ====================\n")
for p in preds:
    print("TEXT:", repr(getattr(p, "text", None)))
    print("RAW :", repr(getattr(p, "prediction", "")))

Image size: (407, 36)

==================== RESULT ====================

TEXT: None
RAW : 'tractat lequal es partida dela operacio amn'


## **3. Formatting the .json that we have as an output from kraken as ALTO XML**
We can use this formatted file to do the transcription in eScriptorium

In [84]:
import os
import json
from pathlib import Path
from PIL import Image
from kraken import serialization
from kraken.containers import Segmentation, BBoxLine, BaselineLine

# Define your project root if not already in environment
# PROJECT_ROOT = "/path/to/your/project"

# Load JSON
def format_from_JSON_to_ALTO_XML(input_json_path,input_img_path, output_alto_path):
    with open(input_json_path, encoding="utf-8") as f:
        data = json.load(f)
    im = Image.open(input_img_path)

    lines = []
    for line in data["lines"]:
        if line.get("type") == "baselines":
            lines.append(BaselineLine(
                id=line["id"],
                baseline=line["baseline"],
                boundary=line["boundary"],
                text=line.get("text"),
                tags=line.get("tags"),
            ))
        else:  
            lines.append(BBoxLine(
                id=line["id"],
                bbox=line["bbox"],
                text=line.get("text"),
                tags=line.get("tags"),
                split=line.get("split"),
            ))

    seg = Segmentation(
        type=data["type"],
        imagename=data["imagename"],
        text_direction=data["text_direction"],
        script_detection=data["script_detection"],
        lines=lines,
        regions=data.get("regions", {}),
        line_orders=data.get("line_orders", [])
    )

    alto = serialization.serialize(seg, image_size=im.size)
    with open(output_alto_path, "w", encoding="utf-8") as f:
        f.write(alto)


In [85]:
from pathlib import Path
from tqdm import tqdm  
import sys
sys.path.insert(0, str(Path(os.environ.get("PROJECT_ROOT", "."))))
from src.utils.path_utils import format_filename, format_for_cli
input_folder = Path(PROJECT_ROOT, "data/raw/original_manuscript/reproduction14453_100")
output_folder = Path(PROJECT_ROOT, "data/processed/segmented_images/segmentation_20260505_172845")

alto_dir = output_folder / "alto_format"
alto_dir.mkdir(parents=True, exist_ok=True)

image_extensions = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
image_files = sorted([f for f in input_folder.iterdir() if f.suffix.lower() in image_extensions])

success_count = 0
for idx, img_path in enumerate(tqdm(image_files, desc="Segmenting", unit="file"), 1):
    base_name = img_path.stem
    output_path, output_filename, processed_name = format_filename(base_name, output_folder)
    input_cmd, output_cmd = format_for_cli(img_path, output_path)
    alto_path = alto_dir / f"{processed_name}.xml"

    format_from_JSON_to_ALTO_XML(input_json_path=output_path,input_img_path=img_path, output_alto_path=alto_path)

Segmenting: 100%|██████████| 71/71 [00:01<00:00, 59.41file/s]


## **4 . Transcribe just 1 text line image -  with scaled baseline** 
When kraken do the segmentation it identifies also the baseline, in this case, for each image we scale that baseline to avoid errors in the OCR prediction

In [ ]:
seg_path= Path(PROJECT_ROOT, "data/processed/segmented_images/segmentation_20260505_172845")

filtered_img_dir= Path(PROJECT_ROOT, "data/processed/filtered_images/20260510_160650/original/kept")
output_dir = Path(PROJECT_ROOT, "data/processed/transcription")
img_inventory = Path(PROJECT_ROOT,"data/processed/filtered_images/20260510_160650/filter_tracking.csv")
model_path = os.path.join(PROJECT_ROOT, "models/ocr/catmus-medieval.mlmodel")
model = models.load_any(model_path, device="cpu")

def bbox_of_polygon(pts):
    xs = [p[0] for p in pts]
    ys = [p[1] for p in pts]
    return min(xs), min(ys), max(xs), max(ys)


filt_inv = pd.read_csv(img_inventory)
filt_inv=filt_inv[filt_inv["was_removed"]==False] 

for img_ in sorted(set(filt_inv.folder)):
    json_name = f"{img_}.json"
    json_path = seg_path / json_name
    with open(json_path, "r", encoding="utf-8") as f:
        json_data = json.load(f)
    json_data_lines = json_data['lines']
    text_lines = sorted(set(filt_inv[filt_inv['folder']== img_]['stem']))
    id_text_lines = [int(stem.split('_')[-1]) for stem in text_lines]

    output_txt_dir = output_dir / img_
    output_txt_dir.mkdir(parents=True, exist_ok=True)

    for idx, img_nm in zip(id_text_lines, text_lines):
        filtered_img_path = filtered_img_dir / img_ / f"{img_nm}.png"
        if not filtered_img_path.exists():
            print(f"Skipping {img_nm}: image not found")
            continue

        if idx < 0 or idx >= len(json_data_lines):
            print(f"Skipping {img_nm}: invalid json index")
            continue

        line_data = json_data_lines[idx]
        x0, y0, x1, y1 = bbox_of_polygon(line_data['boundary'])
        im = Image.open(filtered_img_path).convert("L")
        w, h = im.size
        raw_bl = line_data['baseline']
        local_bl = [(x - x0, y - y0) for x, y in raw_bl]

        clipped_bl = [(max(0, min(w - 1, x)), max(0, min(h - 1, y))) for x, y in local_bl]

        line = BaselineLine(
            id="0",
            baseline=clipped_bl,
            boundary=[(0, 0), (w - 1, 0), (w - 1, h - 1), (0, h - 1)])

        seg = Segmentation(
            type="baselines",
            imagename=str(filtered_img_path),
            text_direction="horizontal-lr",
            script_detection=False,
            lines=[line])

        preds = list(rpred.rpred(model, im, seg))


        output_txt_path = output_txt_dir / f"{img_nm}.txt"
        with open(output_txt_path, "w", encoding="utf-8") as f:
            if preds:
                p = preds[0]
                pred = getattr(p, "prediction", "")
                print("PRED:", repr(pred))
                f.write(pred + "\n")
            else:
                print("NO PREDICTION")
                f.write("\n")

PRED: 'yssi comensan las paraulas de'
PRED: 'de albucasin'
PRED: 'delu so conseqͥt'
PRED: 'per aquo fill necessari es a'
PRED: 'la fe eulu. eper'
PRED: 'las exposicios'
PRED: 'delu. eper las de claracios delu es ami uist'
PRED: 'que yeu complesca aquela a uos an a q̃st'
PRED: 'tractat lequal es partida dela operacio amn'
PRED: 'ma so es corurgia. quar la operaçio am ma'
PRED: 'des prostrada en nostre religio. e en nostie'
PRED: 'temmps de tot priuada entro que fort leu pe'
PRED: 'ric la sciencia delu. ele uistigi delu es ostat'
PRED: 'e no romasero delu sino alcimnas petitas'
PRED: 'descripcios enles libres dels aucies les q̃ls'
PRED: 'nuidero las mas. e en deuenc adaquels'
PRED: 'error. e heyssitacio entro que son clausas'
PRED: 'las entencios delu. ees elonguada la for'
PRED: 'sa delu. clart. ¶ ¶ es uist ami que'
PRED: 'yeu uiuifique aquela ain ordenacio de a'
PRED: 'quest tractat en aquela segon la nia de'
PRED: 'exposicio. ede declaracio. ede abiemacio. e'
PRED: 'perque uengua aul